# 03 Top-3 Confidence and Guidance

Upgrade the output layer from single-label prediction to ranked top-3 diagnoses, confidence-style scores, and cautious consultation guidance.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import SEED, SYNTHEA_DIR
from src.data_prep import build_patient_dataset, filter_top_labels
from src.prompts import prompt_zero_shot_ranked
from src.llm_runner import run_self_consistency_ranked
from src.parsing import parse_ranked_output

patients_df = pd.read_csv(SYNTHEA_DIR / "patients.csv")
conditions_df = pd.read_csv(SYNTHEA_DIR / "conditions.csv")
df = build_patient_dataset(patients_df, conditions_df)
df, LABEL_SPACE = filter_top_labels(df, top_n=4)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["diagnosis"],
)

row = test_df.reset_index(drop=True).iloc[0]
prompt = prompt_zero_shot_ranked(row, LABEL_SPACE)
sc_output = run_self_consistency_ranked(prompt, LABEL_SPACE, parse_ranked_output, n=5)
sc_output
